In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Step 1 — Check Kaggle Environment**

In [2]:
# Check Python version

import sys

print("Python version:", sys.version)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# **Step 2 — Install the libraries**

In [3]:
!pip install -q transformers accelerate

# **Step 3 — Check Transformers**

In [4]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.0.0


# **Step 4 — Check GPU Availability**

In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cpu
GPU available: False


# **Step 5 — Load a Conversational Model**

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # float32 kyunke CPU use ho raha hai
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully on:", device)

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cpu


# **Step 6 — Build the Chatbot Function**

In [9]:
# System prompt — you can change here the chatbot "personality" or business use-case set.
SYSTEM_PROMPT = "You are a helpful, friendly AI assistant for a business. Answer clearly and concisely."

# Conversation history will be here
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def chat(user_message, max_new_tokens=200):
    """Ek user message le kar, model se reply generate karta hai aur history update karta hai."""
    conversation_history.append({"role": "user", "content": user_message})

    # Chat template apply and make prompt for template
    prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # only new generated area here (without input prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

# **Step 7 — Test the Chatbot**

In [10]:
response = chat("Hello! Who are you and what can you help me with?")
print("Bot:", response)

Bot: I am a language model designed to assist in various tasks such as answering questions, providing information, writing essays, composing emails, generating text, and much more. My purpose is to help people find the right words and phrases to express themselves effectively. If there's anything specific you'd like assistance with, feel free to ask!


In [11]:
response = chat("Can you give me 3 tips for improving customer service in a small business?")
print("Bot:", response)

Bot: Certainly! Here are three tips that can significantly improve customer service in a small business:

1. **Be Prepared**: Before your first interaction, prepare a list of common customer inquiries or concerns. This includes phone numbers, email addresses, and any other relevant contact details.

2. **Stay Professional**: Maintain a professional demeanor throughout interactions. Even if it's just a brief chat, make sure to use appropriate language and avoid distractions that might distract customers.

3. **Listen Actively**: When customers do raise their issues, listen attentively without interrupting them. Show empathy by asking clarifying questions and offering solutions. Active listening builds trust and understanding between you and your customers.

These practices not only enhance the customer experience but also demonstrate your commitment to providing excellent service.


# **Step 8 — Interactive Chat Loop**

In [ ]:
print("Chatbot ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break
    reply = chat(user_input)
    print("Bot:", reply)

Chatbot ready! Type 'exit' to stop.



You:  hello


Bot: Hello! How can I assist you today?


You:  how can you assist me today?


Bot: As an AI language model, I'm always here to assist you! Here are some ways I can help you today:

1. **Question Answers**: You can ask me any question you have about a particular topic.
2. **Information Retrieval**: Help me find information on different topics.
3. **Writing Assistance**: Provide writing suggestions based on your requirements.
4. **Email Composing**: Write emails tailored to your needs.
5. **Text Message Management**: Reply to messages from users.
6. **Chatbot Interaction**: Chat with virtual assistants to get answers to queries.
7. **Customer Service**: Offer support through customer service chatbots or live agents.
8. **Content Creation**: Write articles, blogs, or write scripts.
9. **Presentation Skills**: Improve your public speaking skills using my AI tools.
10. **Language Learning**: Teach you new languages or improve existing ones.

Feel free to let me know how I can be of assistance to you today!


You:  i have a different query for you today.


Bot: Sure, feel free to ask your next question, and I'll do my best to assist you.


You:  how can i deploy a chatbot for live checking?
